In [16]:
# --- setup
import os, re
import cv2
import pytesseract
import pandas as pd
from datetime import datetime

TABLE_DIR = "../outputs/tables/"    # input: table crops from step 2
OUT_DIR   = "../outputs/ocr/"       # output: CSV here
os.makedirs(OUT_DIR, exist_ok=True)

# If tesseract is not on PATH, set the full path here:
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

def ocr_text(img_bgr):
    """Run OCR on an image with light preprocessing."""
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (3,3), 0)
    thr  = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY+cv2.THRESH_OTSU)[1]
    cfg = "--oem 3 --psm 6 -l eng"
    return pytesseract.image_to_string(thr, config=cfg)


In [17]:
# --- UK bank row parsing (DD MMM YY)

# matches e.g. "07 Aug 23", "7 Aug23", "16Aug23"
UK_DATE_RE = re.compile(r"\b(\d{1,2})\s*([A-Za-z]{3})\s*(\d{2,4})\b")

# map month short -> number
MONTHS = {m.lower(): i for i,m in enumerate(
    ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"], start=1)}

DEBIT_TYPES  = {"DEB","DD","CD","CARD","POS","ATM","FEE","CHG","FPO"}   # money out
CREDIT_TYPES = {"FPI","CR","CREDIT","SAL","INT"}                       # money in

def clean_amt(s: str):
    s = s.replace("£","").replace(",","").strip()
    if s.startswith("(") and s.endswith(")"):
        s = "-" + s[1:-1]
    s = s.strip(".")
    try: return float(s)
    except: return None

def find_last_two_amounts(line: str):
    toks = line.split()
    nums = []
    for tok in reversed(toks):
        val = clean_amt(tok)
        if val is not None:
            nums.append(val)
            if len(nums) == 2:
                break
    if len(nums) < 2:
        return None, None
    amount, balance = nums[1], nums[0]    # because we scanned right->left
    return amount, balance

def parse_uk_row(line: str):
    m = UK_DATE_RE.search(line)
    if not m:
        return None

    d, mon, y = m.groups()
    d = int(d)
    mon = MONTHS.get(mon.lower())
    if not mon:
        return None
    y = int(y)
    if y < 100:  # 2-digit year
        y += 2000
    try:
        date_obj = datetime(y, mon, d).date()
    except:
        return None

    # after the date = description + type + amounts
    desc_region = line[m.end():].strip()
    desc_region = desc_region.replace("|", " ").replace("  ", " ")

    # detect type token
    type_token = None
    for t in sorted(DEBIT_TYPES | CREDIT_TYPES, key=len, reverse=True):
        if re.search(rf"\b{t}\b", desc_region, flags=re.IGNORECASE):
            type_token = t.upper()
            break

    # extract amount & balance
    amount, balance = find_last_two_amounts(desc_region)
    if amount is None or balance is None:
        return None

    # description = part before the type token (if present)
    description = desc_region
    if type_token:
        pos = description.upper().rfind(type_token)
        if pos > 0:
            description = description[:pos].strip()

    if not description:
        description = "[NO_DESCRIPTION]"

    # decide debit/credit
    debit = credit = None
    if type_token in DEBIT_TYPES:
        debit, credit = amount, None
    elif type_token in CREDIT_TYPES:
        debit, credit = None, amount
    else:
        debit, credit = amount, None

    return date_obj, description, debit, credit, balance


In [18]:
rows = []
table_files = sorted([f for f in os.listdir(TABLE_DIR) if f.lower().endswith((".png",".jpg",".jpeg"))])
print(f"[INFO] Found {len(table_files)} table crops.")

for fn in table_files:
    fp = os.path.join(TABLE_DIR, fn)
    img = cv2.imread(fp)
    if img is None:
        print("[WARN] could not read:", fp)
        continue

    text = ocr_text(img)

    # quick preview
    print(f"\n--- OCR preview: {fn} ---")
    print("\n".join(text.splitlines()[:6]))
    print("-------------------------")

    for raw in text.splitlines():
        line = raw.strip()
        if not line:
            continue
        parsed = parse_uk_row(line)
        if parsed is None:
            continue
        d, desc, debit, credit, bal = parsed
        rows.append({
            "date": d,
            "description": desc,
            "debit": debit,
            "credit": credit,
            "balance": bal,
            "source_image": fn
        })

print("\nrows collected:", len(rows))
if rows:
    print("sample row:", rows[0])


[INFO] Found 21 table crops.

--- OCR preview: 2023_August_Statement (6)_page_1_table_1.png ---
oO oO Oo oOo Oo Oo oO Q oO Oo oO oOo oO oO oOo iw]
Ni Ni - a4 a4 a4 a4 o Ne) Ns) = = = = = bry
> > > > > > > > > > > > > > > Oo
Cc Cc Cc Cc Cc Cc Cc Cc Cc Cc Cc Cc Cc Cc Cc
a a a a ra] ra) ra] ro ro] iro] ira] (ro) a a a
n nN nN nN nN nN nN nN 1) 1) nN nN nN 1) nN
-------------------------

--- OCR preview: 2023_August_Statement (6)_page_2_table_1.png ---
07 Aug23_ | TESCO PAY AT PUMP. CD 0523 | DEB 30.01 10,454.33
08 Aug 23 OO ESSN ot FPI 350.00 10,804.33
09 Aug23 | VANTAGE TOYOTA CD 0523 DEB 288.00 10,516.33
10 Aug 23 | HMRC E VAT 0000148023 DD 960.00 9,556.33
10 Aug 23 | ee NSE FPI 832.00 10,388.33
10 Aug 23 | $Oo900001179221216 RADIO FPO 500.00 9,888.33
-------------------------

--- OCR preview: 2023_August_Statement (6)_page_3_table_1.png ---
16Aug23 | eBay 0°02-10313-37CD0515 | DEB 8.66 10,715.97
16 Aug 23 | eee oacce3paRTs | FPO 3,600.00 7,115.97
17 Aug 23 | Oe DD 160.10 6,955.87
17 

In [19]:
if not rows:
    print("⚠️ No rows parsed. If previews look okay, we can loosen regex or tweak type mapping.")
else:
    df = pd.DataFrame(rows)
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    for c in ["debit","credit","balance"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    out_csv = os.path.join(OUT_DIR, "structured_transactions.csv")
    df.to_csv(out_csv, index=False)
    print("✅ Saved:", out_csv, "| rows:", len(df))
    display(df.head(20))


✅ Saved: ../outputs/ocr/structured_transactions.csv | rows: 191


,date,description,debit,credit,balance,source_image
0,2023-08-08,OO ESSN ot,NaN,350.0,10804.33,2023_August_Statement (6)_page_2_table_1.png
1,2023-08-09,VANTAGE TOYOTA CD 0523,288.00,NaN,10516.33,2023_August_Statement (6)_page_2_table_1.png
2,2023-08-10,HMRC E VAT 0000148023,960.00,NaN,9556.33,2023_August_Statement (6)_page_2_table_1.png
3,2023-08-10,ee NSE,NaN,832.0,10388.33,2023_August_Statement (6)_page_2_table_1.png
4,2023-08-10,$Oo900001179221216 RADIO,500.00,NaN,9888.33,2023_August_Statement (6)_page_2_table_1.png
5,2023-08-10,eBay 0*19-10389-11CD 0515,16.45,NaN,9871.88,2023_August_Statement (6)_page_2_table_1.png
6,2023-08-10,NO a pgoooet 186200789,200.00,NaN,9671.88,2023_August_Statement (6)_page_2_table_1.png
7,2023-08-11,Oe ESS anaat,NaN,250.0,9921.88,2023_August_Statement (6)_page_2_table_1.png
8,2023-08-11,Hac EBK *Q624VSTSG2 CD,31.67,NaN,9890.21,2023_August_Statement (6)_page_2_table_1.png
9,2023-08-14,TESCOEFS 3888 CD 0528,44.40,NaN,9845.81,2023_August_Statement (6)_page_2_table_1.png
